# Fig 3.7.1 — Model validation: population vector correlation, coding geometry, and parameter selection

Figure-only notebook. Loads cached results from fig3_1_1 and fig3_4_1 — no simulations are rerun
unless the model PVC trajectory is not yet cached.

| Panel | Content |
|-------|---------|
| **A** | Population vector correlation vs Rule et al. (2020) |
| **B** | PVC fit score (MSE heatmap) |
| **C** | Coding subspace overlap vs Rule et al. (2020) |
| **D** | Subspace overlap fit score (MSE heatmap) |
| **E** | Combined fit score (log-ratio, full width) |

In [1]:
%matplotlib inline
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit

sys.path.insert(0, os.path.abspath('.'))
from model import ModelParams, get_or_run, extract_arrays, compute_subspace_overlap
from plot_style import apply_style, COLORS, FONT, LW, LINEWIDTH_IN

NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')
FIGURES_DIR  = os.path.join(NOTEBOOK_DIR, 'saved_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

RULE_PVC_CACHE     = os.path.join(DATA_DIR, 'rule_pvc_cache.npz')
GRID_CACHE         = os.path.join(DATA_DIR, 'grid_search_cache.npz')
RULE_OV_CACHE      = os.path.join(DATA_DIR, 'rule_subspace_overlap_cache.npz')
SUBSPACE_CACHE     = os.path.join(DATA_DIR, 'subspace_overlap_cache.npz')
CANONICAL_OV_CACHE = os.path.join(DATA_DIR, 'subspace_overlap_canonical_cache.npz')

for p in [RULE_PVC_CACHE, GRID_CACHE, RULE_OV_CACHE, SUBSPACE_CACHE, CANONICAL_OV_CACHE]:
    tag = 'found' if os.path.isfile(p) else 'MISSING'
    print(f'  {os.path.basename(p)}: {tag}')

  rule_pvc_cache.npz: found
  grid_search_cache.npz: found
  rule_subspace_overlap_cache.npz: found
  subspace_overlap_cache.npz: found
  subspace_overlap_canonical_cache.npz: found


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Model parameters — must match fig3_1_1 / fig3_4_1   ║
# ╚══════════════════════════════════════════════════════════╝
E          = 0.5
vol_std    = 0.01
taur       = 50.0
tauw       = 1000.0
decay      = 1000.0
seeds      = list(range(10))
Nevent     = 7
DAY_MAX    = 7
K_SUBSPACE = 3
nstep      = 3000 * Nevent + 1000

GRID_E       = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
GRID_VOL_STD = [0.0, 0.001, 0.002, 0.005, 0.01, 0.015, 0.02, 0.03]
DRIFT_THRESH = 0.90
MOUSE_IDS    = ['m01', 'm03', 'm04', 'm05']

print(f'E={E}  vol_std={vol_std}  seeds={len(seeds)}  Nevent={Nevent}  K={K_SUBSPACE}')

E=1.0  vol_std=0.005  seeds=10  Nevent=7  K=3


In [ ]:
# ── Rule et al. PVC ───────────────────────────────────────────────────────────
raw = np.load(RULE_PVC_CACHE)
mouse_ids = sorted({k.split('_')[0] for k in raw.keys()})
rule_pvc_curves = {
    mid: dict(days=raw[f'{mid}_days'], vals=raw[f'{mid}_pvcs'])
    for mid in mouse_ids
}

def _exp_decay(t, tau):
    return np.exp(-(t - 1.0) / tau)

rule_pvc_taus = {}
for mid, curve in rule_pvc_curves.items():
    days  = curve['days'].astype(float)
    pvcs  = curve['vals']
    valid = ~np.isnan(pvcs) & (days >= 1)
    if valid.sum() < 3:
        rule_pvc_taus[mid] = np.nan; continue
    try:
        popt, _ = curve_fit(_exp_decay, days[valid], pvcs[valid],
                             p0=[20.0], bounds=(0.5, 2000.0))
        rule_pvc_taus[mid] = float(popt[0])
    except Exception:
        rule_pvc_taus[mid] = np.nan

# ── Rule et al. subspace overlap ──────────────────────────────────────────────
with np.load(RULE_OV_CACHE) as _f:
    rule_ov_raw = {
        mid: dict(days=_f[f'{mid}_days'].copy(), vals=_f[f'{mid}_ov'].copy())
        if f'{mid}_days' in _f else None
        for mid in MOUSE_IDS
    }

# Sort each mouse's OV data by day
for mid, data in rule_ov_raw.items():
    if data is None: continue
    order = np.argsort(data['days'])
    data['days'] = data['days'][order]
    data['vals'] = data['vals'][order]

# Day 1 overlap with itself is 1.0 by definition; prepend if absent from digitised data
for mid, data in rule_ov_raw.items():
    if data is None: continue
    if not np.any(np.isclose(data['days'], 1.0)):
        data['days'] = np.concatenate([[1.0], data['days']])
        data['vals'] = np.concatenate([[1.0], data['vals']])

# Pool by integer day for the MSE computation (Day 1 excluded — always 1.0)
rule_ov_by_day = {d: [] for d in range(2, DAY_MAX + 1)}
for data in rule_ov_raw.values():
    if data is None: continue
    for day, ov in zip(data['days'], data['vals']):
        d = int(round(day))
        if 2 <= d <= DAY_MAX:
            rule_ov_by_day[d].append(ov)

rule_ov_mean = {d: float(np.mean(v)) if v else np.nan for d, v in rule_ov_by_day.items()}

# ── Grid caches ───────────────────────────────────────────────────────────────
with np.load(GRID_CACHE) as _gc:
    mse_pvc      = _gc['loss_grid'].copy()
    best_pvc_E   = float(_gc['best_E'])
    best_pvc_vol = float(_gc['best_vol'])

with np.load(SUBSPACE_CACHE) as _sw:
    ov_final_grid = _sw['ov_final'].copy()
    ov_traj_grid  = _sw['ov_traj'].copy()

invalid_mask = ov_final_grid >= DRIFT_THRESH

# OV MSE vs Rule et al.
comp_days   = [d for d in range(2, DAY_MAX + 1)
               if not np.isnan(rule_ov_mean.get(d, np.nan))]
ov_mse_grid = np.full((nE, nV), np.nan)
for ii in range(nE):
    for jj in range(nV):
        traj = ov_traj_grid[ii, jj]
        sq   = [(traj[d-1] - rule_ov_mean[d])**2 for d in comp_days
                if not (np.isnan(traj[d-1]) or np.isnan(rule_ov_mean[d]))]
        if sq: ov_mse_grid[ii, jj] = float(np.mean(sq))

# Combined log-ratio score
def _log_ratio(x):
    return np.log(np.clip(x / np.nanmin(x), 1.0, None))

combined      = _log_ratio(mse_pvc) + _log_ratio(ov_mse_grid)
combined_norm = combined / np.nanmax(combined)
best_c = np.unravel_index(
    np.nanargmin(np.where(invalid_mask, np.nan, combined)), combined.shape)

print(f'Caches loaded.')
print(f'Best PVC fit:      E={best_pvc_E}   vol_std={best_pvc_vol}')
best_ov_idx = np.unravel_index(
    np.nanargmin(np.where(invalid_mask, np.nan, ov_mse_grid)), ov_mse_grid.shape)
print(f'Best OV fit:       E={GRID_E[best_ov_idx[0]]}   vol_std={GRID_VOL_STD[best_ov_idx[1]]}')
print(f'Best combined fit: E={GRID_E[best_c[0]]}   vol_std={GRID_VOL_STD[best_c[1]]}')

In [4]:
# ── Model PVC trajectory ──────────────────────────────────────────────────────────────
def _model_stim_map(result):
    r   = extract_arrays(result)['r']
    seq = result['seq']
    p   = result['params']
    maps = []
    for ev in range(p.Nevent):
        sm = [r[:, seq[ev*2*p.Nstim+2*i]:seq[ev*2*p.Nstim+2*i+1]].mean(axis=1)
              for i in range(p.Nstim)]
        maps.append(np.stack(sm, axis=1))
    return maps

def _pvc_from_maps(maps):
    ref = maps[0].ravel()
    return [1.0] + [float(np.corrcoef(ref, maps[k].ravel())[0, 1])
                    for k in range(1, len(maps))]

all_seed_pvcs = []
for seed in seeds:
    p   = ModelParams(E=E, vol_std=vol_std, taur=taur, tauw=tauw,
                      decay=decay, seed=seed, Nevent=Nevent, nstep=nstep)
    res = get_or_run(p, '')
    all_seed_pvcs.append(_pvc_from_maps(_model_stim_map(res)))

pvc_arr        = np.array(all_seed_pvcs)
model_pvc_mean = pvc_arr.mean(axis=0)
model_pvc_sem  = pvc_arr.std(axis=0) / np.sqrt(len(seeds))
model_days     = np.arange(1, Nevent + 1)

try:
    popt_m, _ = curve_fit(_exp_decay, model_days.astype(float), model_pvc_mean,
                           p0=[20.0], bounds=(0.5, 2000.0))
    model_pvc_tau = float(popt_m[0])
except Exception:
    model_pvc_tau = np.nan

# ── Model subspace overlap trajectory ────────────────────────────────────────────────
def _canonical_ov_valid():
    if not os.path.isfile(CANONICAL_OV_CACHE): return False
    d = np.load(CANONICAL_OV_CACHE)
    return (float(d['E_c'])  == E         and float(d['vol_c']) == vol_std and
            int(d['K_c'])    == K_SUBSPACE and int(d['Nev_c'])  == Nevent)

if _canonical_ov_valid():
    _c = np.load(CANONICAL_OV_CACHE)
    model_ov_mean = _c['ov_mean']
    model_ov_sem  = _c['ov_sem']
    print('Subspace overlap: loaded canonical cache.')
else:
    all_seed_ov = []
    for seed in seeds:
        p   = ModelParams(E=E, vol_std=vol_std, taur=taur, tauw=tauw,
                          decay=decay, seed=seed, Nevent=Nevent, nstep=nstep)
        res = get_or_run(p, '')
        all_seed_ov.append(compute_subspace_overlap(res, k=K_SUBSPACE))
    ov_arr        = np.array(all_seed_ov)
    model_ov_mean = ov_arr.mean(axis=0)
    model_ov_sem  = ov_arr.std(axis=0) / np.sqrt(len(seeds))
    np.savez(CANONICAL_OV_CACHE,
             ov_mean=model_ov_mean, ov_sem=model_ov_sem,
             E_c=np.float64(E), vol_c=np.float64(vol_std),
             K_c=np.int64(K_SUBSPACE), Nev_c=np.int64(Nevent))
    print('Subspace overlap: computed and cached.')

print(f'Model PVC \u03c4 = {model_pvc_tau:.1f}d')
print(f'Model OV (days 1-7): {chr(32).join(f"{v:.3f}" for v in model_ov_mean)}')

Running simulation (seed=0, E=1.0, vol_std=0.005) ...
Running simulation (seed=1, E=1.0, vol_std=0.005) ...
Running simulation (seed=2, E=1.0, vol_std=0.005) ...
Running simulation (seed=3, E=1.0, vol_std=0.005) ...
Running simulation (seed=4, E=1.0, vol_std=0.005) ...
Running simulation (seed=5, E=1.0, vol_std=0.005) ...
Running simulation (seed=6, E=1.0, vol_std=0.005) ...
Running simulation (seed=7, E=1.0, vol_std=0.005) ...
Running simulation (seed=8, E=1.0, vol_std=0.005) ...
Running simulation (seed=9, E=1.0, vol_std=0.005) ...
Subspace overlap: loaded canonical cache.
Model PVC τ = 8.8d
Model OV (days 1-7): 1.000 0.619 0.436 0.398 0.325 0.393 0.292


In [ ]:
import matplotlib as _mpl

RULE_COLOR     = '#cc3333'
MODEL_COLOR    = COLORS.get('exc', '#1f77b4')
DATA_LW        = LW['trace']
_MOUSE_MARKERS = {'m01': 'o', 'm03': 's', 'm04': '^', 'm05': 'D'}


def _trajectory(ax, rule_curves, rule_taus_dict, model_mean, model_sem,
                model_tau_val, ylabel, title, panel_letter):
    for mid, curve in rule_curves.items():
        if curve is None: continue
        days = curve['days']
        vals = curve['vals']
        mask = (~np.isnan(vals)) & (days <= DAY_MAX)
        if mask.sum() == 0: continue
        d_plot, v_plot = days[mask], vals[mask]
        tau    = rule_taus_dict.get(mid, np.nan)
        taustr = f' (τ={tau:.0f}d)' if not np.isnan(tau) else ''
        marker = _MOUSE_MARKERS.get(mid, 'o')
        ax.plot(d_plot, v_plot, color=RULE_COLOR, lw=DATA_LW,
                linestyle=':', alpha=0.65, marker=marker, markersize=4)
        ax.annotate(mid.upper() + taustr,
                    xy=(d_plot[-1], v_plot[-1]),
                    xytext=(4, 0), textcoords='offset points',
                    va='center', ha='left',
                    fontsize=FONT.get('annotation', 8) - 1,
                    color=RULE_COLOR)
    ax.plot([], [], color=RULE_COLOR, lw=DATA_LW, linestyle=':', marker='o',
            markersize=4, alpha=0.65, label='Rule et al. (2020)')

    tau_str_m = f', τ={model_tau_val:.0f}d' if not np.isnan(model_tau_val) else ''
    mask_m = model_days <= DAY_MAX
    ax.fill_between(model_days[mask_m],
                    (model_mean - model_sem)[mask_m],
                    (model_mean + model_sem)[mask_m],
                    color=MODEL_COLOR, alpha=0.20)
    ax.plot(model_days[mask_m], model_mean[mask_m],
            color=MODEL_COLOR, lw=DATA_LW * 1.8, linestyle='--',
            marker='s', markersize=5,
            label=f'Model (E={E}, σ={vol_std}{tau_str_m})')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.4)
    ax.set_xlim(0.5, DAY_MAX + 1.5)
    ax.set_xticks(np.arange(1, DAY_MAX + 1))
    ax.set_ylim(-0.05, 1.10)
    ax.set_xlabel('Day')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=FONT.get('annotation', 8) - 1, loc='lower left')
    ax.text(-0.15, 1.02, panel_letter, transform=ax.transAxes,
            fontsize=FONT.get('panel_label', 13), fontweight='bold')


def _heatmap(ax, grid, cbar_label, title, panel_letter,
             best_E_val=None, best_vol_val=None, show_model_star=True,
             hatch_invalid=False):
    _cmap = _mpl.colormaps['viridis'].copy()
    _cmap.set_bad('lightgrey')
    im = ax.imshow(grid, origin='lower', aspect='auto', cmap=_cmap,
                   vmin=0.0, vmax=float(np.nanmax(grid)),
                   extent=[-0.5, nV - 0.5, -0.5, nE - 0.5])
    ax.set_xticks(range(nV))
    ax.set_xticklabels([str(v) for v in GRID_VOL_STD], rotation=45, ha='right',
                        fontsize=FONT.get('annotation', 8) - 1)
    ax.set_yticks(range(nE))
    ax.set_yticklabels([str(e) for e in GRID_E],
                        fontsize=FONT.get('annotation', 8) - 1)
    ax.set_xlabel('Synaptic volatility (vol_std)')
    ax.set_ylabel('Excitability amplitude (E)')
    ax.set_title(title)
    cb = ax.figure.colorbar(im, ax=ax)
    cb.set_label(cbar_label, fontsize=FONT.get('annotation', 8) - 1)
    if hatch_invalid:
        for ii in range(nE):
            for jj in range(nV):
                if invalid_mask[ii, jj]:
                    ax.add_patch(plt.Rectangle(
                        [jj-0.5, ii-0.5], 1, 1,
                        fill=True, facecolor='lightgrey', alpha=0.55,
                        hatch='///', edgecolor='grey', linewidth=0.4, zorder=3))
    if best_E_val is not None:
        b_i = int(np.where(np.isclose(np.array(GRID_E),       best_E_val))[0][0])
        b_j = int(np.where(np.isclose(np.array(GRID_VOL_STD), best_vol_val))[0][0])
        ax.plot(b_j, b_i, 'r*', markersize=12,
                label=f'Best: E={best_E_val}, σ={best_vol_val}')
    if show_model_star and E in GRID_E and vol_std in GRID_VOL_STD:
        ax.plot(GRID_VOL_STD.index(vol_std), GRID_E.index(E), 'w*',
                markersize=10, markeredgecolor='black', markeredgewidth=0.5,
                label=f'Model: E={E}, σ={vol_std}')
    ax.legend(fontsize=FONT.get('annotation', 8) - 1)
    ax.text(-0.15, 1.02, panel_letter, transform=ax.transAxes,
            fontsize=FONT.get('panel_label', 13), fontweight='bold')


# ── Fig 3.7.1 — 4-panel figure ───────────────────────────────────────────────
apply_style()

fig, axes = plt.subplots(2, 2, figsize=(LINEWIDTH_IN, 6.5))
ax_a, ax_b = axes[0]
ax_c, ax_d = axes[1]

# Panel A — PVC trajectory
_trajectory(ax_a, rule_pvc_curves, rule_pvc_taus,
            model_pvc_mean, model_pvc_sem, model_pvc_tau,
            ylabel='Population vector correlation\n(vs Day 1)',
            title='Population vector stability',
            panel_letter='A')

# Panel B — PVC MSE heatmap
_heatmap(ax_b, mse_pvc,
         cbar_label='MSE',
         title='PVC fit score',
         panel_letter='B',
         best_E_val=best_pvc_E, best_vol_val=best_pvc_vol,
         hatch_invalid=False)

# Panel C — subspace overlap trajectory
_trajectory(ax_c, rule_ov_raw, {mid: np.nan for mid in MOUSE_IDS},
            model_ov_mean, model_ov_sem, np.nan,
            ylabel=f'Subspace overlap (k={K_SUBSPACE})',
            title='Coding geometry stability',
            panel_letter='C')

# Panel D — subspace overlap MSE heatmap
_heatmap(ax_d, ov_mse_grid,
         cbar_label='MSE',
         title='Subspace overlap fit score',
         panel_letter='D',
         best_E_val=GRID_E[best_ov_idx[0]], best_vol_val=GRID_VOL_STD[best_ov_idx[1]],
         hatch_invalid=True)

plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'fig3_7_1_combined.pdf')
fig.savefig(out, dpi=300, bbox_inches='tight')
print('Saved to', out)
plt.show()